# Exp 9 - individual-fish responses

Based on the shared single-fish workflow. Presentation timestamps come from this fish's block log. Explicit trajectory metadata defines preconditioning, blank delay and bouts; C1 has no preconditioning. B1-B4 share imaging/stimulus triggers; B0 and interblock waiting are excluded. Sampling is 2 Hz (imaging) and 60 Hz (projection). This workflow does not correct unidentified dropped imaging frames.

After updating plotting helpers, rerun the imports cell below and then the desired plotting cells. The imports cell refreshes the plotting source without clearing existing analysis results.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data_loading.py').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import src.data_loading as data_loading
import src.trial_alignment as trial_alignment
import src.response_normalization as response_normalization
import src.reliability as reliability
import src.plotting_single_fish as plotting

# Refresh plotting from source when reusing a kernel after helper edits.
# Direct compilation avoids retaining an older cached function definition.
PLOTTING_SOURCE = REPO_ROOT / 'src' / 'plotting_single_fish.py'
exec(compile(PLOTTING_SOURCE.read_text(encoding='utf-8-sig'),
             str(PLOTTING_SOURCE), 'exec'), vars(plotting))
import src.significant_trace_detection as significant

## Recording and display settings

Choose a session explicitly if the fish has multiple sessions. Display windows are adjustable; they are not phase-specific measurement windows. Detector tuning lives beside the significant-activity computation below. Nothing is saved until the final save cell is enabled. `SOCIAL_FILTERS_OUTPUT_ROOT` can redirect outputs for validation.


In [ ]:
FISH_ID = 'L978_f02'
EXPERIMENT_NAME = 'Exp_9_RF_memory_inhibition'
SESSION_ID = None
LAB_ROOT = Path('D:/Alejandro/Data/OneDrive - Universit\u00e9 de Lausanne/Lab')
MAIN_PATH = LAB_ROOT / 'Data' / '2p'
STIMULI_MAIN_PATH = LAB_ROOT / 'Data' / 'stimuli'
PHASE_METADATA_PATH = LAB_ROOT / 'Results' / 'Paper' / '2p_derived' / EXPERIMENT_NAME / 'stimuli' / 'parameters' / 'stimulus_time_summary.csv'
SELECTED_BLOCKS = ['B1', 'B2', 'B3', 'B4']
STIMULUS_ALIASES = {
    'LeC1': 'LeC1_Bout', 'LeC2': 'LeC2_Far',
    'LeS1': 'LeS1_Overlap', 'LeS2': 'LeS2_Overlap', 'LeS3': 'LeS3_Overlap',
    'LeS4': 'LeS4_Overlap', 'LeS5': 'LeS5_Overlap',
    'RiC1': 'RiC1_Bout', 'RiC2': 'RiC2_Far',
    'RiS1': 'RiS1_Overlap', 'RiS2': 'RiS2_Overlap', 'RiS3': 'RiS3_Overlap',
    'RiS4': 'RiS4_Overlap', 'RiS5': 'RiS5_Overlap',
}
BASELINE_S = 10.0
DISPLAY_PRE_S = 5.0
RECOVERY_S = 10.0
SORT_MODE = 'corravg'
RANDOM_STATE = 42
Z_LIMITS = (-1.0, 4.0)
RELIABILITY_THRESHOLD = None  # None uses the existing automatic Otsu threshold.
OUTPUT_ROOT = Path(os.environ['SOCIAL_FILTERS_OUTPUT_ROOT']) if os.environ.get('SOCIAL_FILTERS_OUTPUT_ROOT') else None

## Load and audit the recording

Missing trials, unknown names, inconsistent session identities and incompatible frame counts stop the analysis. Logged playback durations are compared with nominal trajectory durations without rescaling phases.


In [ ]:
loaded = data_loading.load_phase_aligned_experiment(
    FISH_ID, EXPERIMENT_NAME, MAIN_PATH, STIMULI_MAIN_PATH,
    PHASE_METADATA_PATH, STIMULUS_ALIASES, SELECTED_BLOCKS,
    session_id=SESSION_ID, baseline_s=BASELINE_S)
dfof, fps_2p = loaded['dfof'], loaded['fps_2p']
trials = loaded['trials']
STIMULUS_KEYS = list(STIMULUS_ALIASES.values())
STIMULUS_IDS = [loaded['stimuli_id_map'][key] for key in STIMULUS_KEYS]
print(f"{FISH_ID}: session {loaded['session_id']}; dFoF {dfof.shape}; {fps_2p:g} Hz")
display(loaded['audit'].round(4))

## Stimulus phases and continuous recording

External pauses come from the recording logs. The phase overview excludes them. Blue boundaries divide concatenated imaging blocks; the bottom strip shows phase timing on each side.


In [ ]:
fig, ax = plotting.plot_phase_timing(trials, STIMULUS_KEYS, figsize=(12, 6))
plt.show()
fig, axes = plotting.plot_continuous_phase_raster(
    dfof, fps_2p, trials, figsize=(17, 5), title=f'{FISH_ID} - continuous dFoF')
plt.show()

## Presentation-based normalization and event alignment

The ten-second baseline precedes the entire presentation, including preconditioning where present. Event anchors are rounded to the nearest imaging sample without interpolation (at most 0.25 s at 2 Hz). Unrounded timestamps and per-trial sample offsets are retained; 0.3/0.6 s delays cannot be finely resolved at this imaging rate.


In [ ]:
sequence_raw = trial_alignment.build_event_aligned_traces(
    dfof, trials, fps_2p, alignment='sequence',
    t_pre_s=DISPLAY_PRE_S, recovery_s=RECOVERY_S)
zres = response_normalization.zscore_dfof_from_prestim_baseline(
    dfof, sequence_raw['onsets_by_id'], fps_2p,
    baseline_s=BASELINE_S, verbose=False)
z_traces = zres['z_traces']
sequence = trial_alignment.build_event_aligned_traces(
    z_traces, trials, fps_2p, alignment='sequence',
    t_pre_s=DISPLAY_PRE_S, recovery_s=RECOVERY_S)
bout = trial_alignment.build_event_aligned_traces(
    z_traces, trials, fps_2p, alignment='bout',
    t_pre_s=DISPLAY_PRE_S, recovery_s=RECOVERY_S)
mean_traces, neuron_order = plotting.prepare_phase_response_display(
    sequence, STIMULUS_IDS, sort_mode=SORT_MODE, random_state=RANDOM_STATE)
print(f"{len(zres['valid_onsets'])} baseline windows; max anchor error "
      f"{max(np.max(np.abs(x)) for x in sequence['alignment_error_s'].values()):.3f} s")

## Full sequences - all neurons

Heatmaps show each neuron's mean across repetitions. The complementary line panels average those trial-mean traces across exactly all neurons shown in the corresponding raster, giving every neuron equal weight. Lines show population mean z-score without offsets; all conditions share the same y-axis. NaNs are omitted. All conditions and both alignments reuse the same neuron order. Empty space after shorter sequences is not data. Shading marks nominal phase boundaries; retained sample offsets describe subframe uncertainty.


In [ ]:
fig, trace_fig = plotting.plot_phase_aligned_responses(
    sequence, STIMULUS_KEYS, STIMULUS_IDS, neuron_order,
    figsize=(19, 8), trace_figsize=(19, 6), trace_mode="population", vmin=Z_LIMITS[0], vmax=Z_LIMITS[1],
    title=f'{FISH_ID} - all neurons')
plt.show()

## Bout comparison - all neurons

Bout onset is zero for C1 and every preconditioned condition. The full-sequence view above retains the complete conditioning history; this view focuses on the five seconds before the bout and its recovery.


In [ ]:
fig, trace_fig = plotting.plot_phase_aligned_responses(
    bout, STIMULUS_KEYS, STIMULUS_IDS, neuron_order,
    figsize=(19, 8), trace_figsize=(19, 6), trace_mode="population", vmin=Z_LIMITS[0], vmax=Z_LIMITS[1],
    title=f'{FISH_ID} - all neurons')
plt.show()

## Whole-sequence reliability

Use the original mean pairwise trial-correlation method, maximum across conditions and automatic Otsu threshold. Each condition's window includes the full sequence, five seconds before and ten seconds after. This selects repeatable sequence responses, not specifically bout-responsive neurons.


In [ ]:
reliability_results = reliability.filter_neurons_by_trial_reliability(
    dfof, sequence['trial_aligned_traces'], STIMULUS_IDS, fps_2p,
    loaded['paths']['plots_path'], prefix=loaded['paths']['prefix'],
    custom_threshold=RELIABILITY_THRESHOLD, make_plots=False, save_indices=False)
kept = reliability_results['kept_neuron_indices']
if not len(kept):
    raise ValueError('No neurons pass whole-sequence reliability; inspect settings before continuing.')
print(f"Kept {len(kept)}/{dfof.shape[1]} neurons; threshold {reliability_results['otsu_threshold']:.3f}")

## Filtered responses

Preserve the all-neuron order and raster z-score limits. Each complementary line panel averages exactly the retained neurons shown in its raster; it uses the same phase shading and time axis. Neurons excluded by reliability do not contribute to this population mean.


In [ ]:
fig, trace_fig = plotting.plot_phase_aligned_responses(
    sequence, STIMULUS_KEYS, STIMULUS_IDS, neuron_order, neuron_indices=kept,
    figsize=(19, 8), trace_figsize=(19, 6), trace_mode="population", vmin=Z_LIMITS[0], vmax=Z_LIMITS[1],
    title=f'{FISH_ID} - reliable neurons')
plt.show()
fig, trace_fig = plotting.plot_phase_aligned_responses(
    bout, STIMULUS_KEYS, STIMULUS_IDS, neuron_order, neuron_indices=kept,
    figsize=(19, 8), trace_figsize=(19, 6), trace_mode="population", vmin=Z_LIMITS[0], vmax=Z_LIMITS[1],
    title=f'{FISH_ID} - reliable neurons')
plt.show()

## Tune significant-activity detection

Edit `SIG_SETTINGS` below for this fish, then rerun detection and the review plots below until satisfied. No detection/review cell writes files. The detector uses every reliability-retained neuron, including zero-event neurons. Diagnostic sorting and example selection affect display only.

Compare original dFoF, noise-centered dFoF and binary event frames before using the final save cell. Red event markers indicate detected calcium transients, not inferred spikes. Keep the same fish and reliability selection while tuning.


In [ ]:
SIG_SETTINGS = dict(
    n_bins=2200, k_neighbors=100, confCutOff=90,
    plot_odds=False, tauDecay=6.0, random_state=0, mode='current',
)
# Snapshot the run only after detection succeeds.
next_sig_settings = dict(SIG_SETTINGS)
next_sig_neurons = kept.copy()
next_sig_results = significant.compute_noise_model_romano_fast_modular(
    dfof[:, next_sig_neurons], fps=fps_2p, **next_sig_settings)
sig_run = dict(fish=loaded['paths']['fish'], session_id=loaded['session_id'],
    source_file=str(loaded['paths']['dfof_file']), fps=fps_2p,
    settings=next_sig_settings, neuron_indices=next_sig_neurons.tolist())
sig_results = next_sig_results
sig_sequence = trial_alignment.build_event_aligned_traces(
    sig_results[6], trials, fps_2p, alignment='sequence',
    t_pre_s=DISPLAY_PRE_S, recovery_s=RECOVERY_S)
sig_bout = trial_alignment.build_event_aligned_traces(
    sig_results[6], trials, fps_2p, alignment='bout',
    t_pre_s=DISPLAY_PRE_S, recovery_s=RECOVERY_S)
sig_order = np.array([np.flatnonzero(kept == n)[0] for n in neuron_order if n in set(kept)])
print(f"Detected {np.count_nonzero(sig_results[6])} event frames in {len(kept)} retained neurons.")


### Compare dFoF and event rasters

All three panels contain the same retained neurons and the same time samples. Original and centered dFoF share display limits. Change color limits or zoom without recomputing detection. Saved arrays stay in retained-column order.


In [ ]:
SIG_DFF_LIMITS = (0.0, 0.05)
SIG_RASTER_WINDOW_S = None  # e.g. (100, 200); display only.
fig_sig_comparison, _, sig_display_ids = plotting.plot_significant_trace_comparison(
    dfof[:, sig_run['neuron_indices']], sig_results[1], sig_results[6],
    sig_run['fps'], sig_run['neuron_indices'], figsize=(14, 9),
    vmin=SIG_DFF_LIMITS[0], vmax=SIG_DFF_LIMITS[1],
    time_window_s=SIG_RASTER_WINDOW_S, title=f'{FISH_ID} - retained neurons')
sig_comparison_run = sig_run
plt.show()

### Same comparison, sorted by original dFoF peak

The same display permutation is applied to all three panels. No neurons are excluded and saved arrays are not reordered.


In [ ]:
fig_sig_peak, _, sig_peak_order_ids = plotting.plot_significant_trace_comparison(
    dfof[:, sig_run['neuron_indices']], sig_results[1], sig_results[6],
    sig_run['fps'], sig_run['neuron_indices'], sort_by_peak=True, figsize=(14, 9),
    vmin=SIG_DFF_LIMITS[0], vmax=SIG_DFF_LIMITS[1],
    time_window_s=SIG_RASTER_WINDOW_S, title=f'{FISH_ID} - original-peak order')
sig_peak_run = sig_run
plt.show()

### Inspect an individual neuron

`NEURON_ID` is its original zero-based merged-neuron ID, not its position in the filtered raster. Choose an ID from `sig_run["neuron_indices"]`. Red dots mark every detected event frame.


In [ ]:
NEURON_ID = sig_run['neuron_indices'][0]  # Replace with a retained original ID.
SIG_TRACE_WINDOW_S = None  # e.g. (100, 200).
fig_sig_neuron, _, sig_single_ids = plotting.plot_significant_trace_examples(
    dfof[:, sig_run['neuron_indices']], sig_results[1], sig_results[6],
    sig_run['fps'], sig_run['neuron_indices'], neuron_ids=[NEURON_ID],
    time_window_s=SIG_TRACE_WINDOW_S, figsize=(13, 3.5),
    title=f'{FISH_ID} - individual retained neuron')
sig_neuron_run = sig_run
plt.show()

### Random examples from high-activity retained neurons

As in the all-neuron review notebook, select up to 20 examples from the top 10% by positive original dFoF peak. This selects examples for display only; it does not change the reliability filter, detector input or output population. Titles show original merged-neuron IDs.


In [ ]:
SIG_N_EXAMPLES = 20
SIG_HIGH_ACTIVITY_FRACTION = 0.10
SIG_EXAMPLE_SEED = 42
SIG_EXAMPLE_WINDOW_S = None
fig_sig_examples, _, sig_example_ids = plotting.plot_significant_trace_examples(
    dfof[:, sig_run['neuron_indices']], sig_results[1], sig_results[6],
    sig_run['fps'], sig_run['neuron_indices'], n_examples=SIG_N_EXAMPLES,
    high_activity_fraction=SIG_HIGH_ACTIVITY_FRACTION, seed=SIG_EXAMPLE_SEED,
    time_window_s=SIG_EXAMPLE_WINDOW_S, figsize=(16, 14),
    title=f'{FISH_ID} - high-activity examples (display only)')
sig_examples_run = sig_run
plt.show()

### Significant activity aligned to stimulus phases

These plots use the current detector result. Re-run this cell after changing detector parameters. Trial means are activation fractions, not phase-specific significance tests.


In [ ]:
fig, _ = plotting.plot_phase_aligned_responses(
    sig_sequence, STIMULUS_KEYS, STIMULUS_IDS, sig_order, neuron_labels=kept,
    figsize=(19, 8), binary=True, trace_count=0, title=f'{FISH_ID} - significant activity')
plt.show()
fig, _ = plotting.plot_phase_aligned_responses(
    sig_bout, STIMULUS_KEYS, STIMULUS_IDS, sig_order, neuron_labels=kept,
    figsize=(19, 8), binary=True, trace_count=0, title=f'{FISH_ID} - significant activity')
plt.show()

## Save only after reviewing this fish

Set `SAVE_OUTPUTS = True` in the next cell and run it when satisfied. Until then no files are written. Changing detector parameters, fish/session or retained neuron IDs after detection makes the save fail; rerun detection and review first.

The usual filtered-output folders and filenames are preserved. The raster and centered dFoF contain only reliability-retained neurons in the same order as `filtered_neurons_by_stimuli/{prefix}_kept_neuron_indices.npy`. The significant NPZ additionally stores that mapping, imaging rate, source and actual detector settings. It is deliberately distinct from the reference notebook's `significant_traces_all` exports.

Save the four review figures beside the significant cache, and record display/example choices in the configuration. Phase-specific measurements remain deferred.


In [ ]:
SAVE_OUTPUTS = False  # Set True only after reviewing this fish.
configuration = dict(baseline_s=BASELINE_S, display_pre_s=DISPLAY_PRE_S,
    recovery_s=RECOVERY_S, sort_mode=SORT_MODE, random_state=RANDOM_STATE,
    trace_mode="population", z_limits=Z_LIMITS,
    reliability_window='whole sequence plus display baseline/recovery',
    reliability_threshold_override=RELIABILITY_THRESHOLD, significant=dict(SIG_SETTINGS), significant_review=dict(
        dff_limits=SIG_DFF_LIMITS, raster_window_s=SIG_RASTER_WINDOW_S,
        inspected_neuron_ids=sig_single_ids.tolist(), trace_window_s=SIG_TRACE_WINDOW_S,
        example_ids=sig_example_ids.tolist(), n_examples=SIG_N_EXAMPLES,
        high_activity_fraction=SIG_HIGH_ACTIVITY_FRACTION, seed=SIG_EXAMPLE_SEED,
        example_window_s=SIG_EXAMPLE_WINDOW_S,
        peak_order_ids=sig_peak_order_ids.tolist()))
if SAVE_OUTPUTS:
    if any(review is not sig_run for review in
           (sig_comparison_run, sig_peak_run, sig_neuron_run, sig_examples_run)):
        raise ValueError('Detector was rerun: refresh all four review plots before saving.')
    provenance_path = data_loading.save_phase_analysis_outputs(
        loaded, zres, reliability_results, sig_results, configuration,
        output_root=OUTPUT_ROOT, significant_run=sig_run,
        review_figures={
            f'{FISH_ID}_dfof_raster_comparison.png': fig_sig_comparison,
            f'{FISH_ID}_dfof_raster_peak_sorted.png': fig_sig_peak,
            f'{FISH_ID}_neuron_{NEURON_ID}_comparison.png': fig_sig_neuron,
            f'{FISH_ID}_high_activity_examples.png': fig_sig_examples,
        })
    print(f'Saved caches and provenance: {provenance_path}')
else:
    print("Review complete; SAVE_OUTPUTS is False, so no files were written.")
